# Lab 07 — Estimación de Fase Cuántica (QPE) Guiada

La Estimación de Fase Cuántica (Quantum Phase Estimation) extrae el valor propio de un operador unitario U.
Si U|ψ⟩ = e^{2πiφ}|ψ⟩, QPE determina φ con precisión de t bits usando t qubits de conteo.

**Aplicaciones**: HHL, Shor, VQE, simulación cuántica.

## 1. Principio de QPE

El circuito QPE tiene tres registros:
1. **Conteo** (t qubits): inicializado en |0⟩^t, captura la fase
2. **Eigenestado** (n qubits): inicializado en |ψ⟩, autovector de U

Pasos:
1. Hadamard en todos los qubits de conteo
2. Puertas U^{2^k} controladas (k = 0, 1, …, t-1)
3. QFT inversa en el registro de conteo
4. Medición → φ ≈ j/2^t donde j es el entero medido

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import QFT
from qiskit.quantum_info import Statevector

def qpe_circuit(phase: float, t: int) -> QuantumCircuit:
    """QPE para U = Rz(2*pi*phase). Eigenestado |+⟩ con eigenvalor e^{i*pi*phase}."""
    counting = QuantumRegister(t, 'count')
    eigenstate = QuantumRegister(1, 'eig')
    cr = ClassicalRegister(t, 'c')
    qc = QuantumCircuit(counting, eigenstate, cr)

    # Preparar eigenestado |+⟩ de Rz
    qc.h(eigenstate[0])

    # Hadamard en registro de conteo
    qc.h(counting)

    # Puertas controladas U^{2^k}
    for k in range(t):
        angle = 2 * np.pi * phase * (2 ** k)
        qc.cp(angle, counting[k], eigenstate[0])

    # QFT inversa en registro de conteo
    qft_inv = QFT(t, inverse=True, do_swaps=True)
    qc.append(qft_inv, counting)

    qc.measure(counting, cr)
    return qc

# Ejemplo con phase = 0.25 (φ = 1/4, esperamos medir j=2^t/4)
phase = 0.25
t = 4
qc = qpe_circuit(phase, t)
print(qc.draw('text'))

## 2. Simulación con StatevectorSampler

Ejecutamos el circuito y extraemos la distribución de medidas. Para t bits de conteo, la estimación de fase es φ̂ = j / 2^t.

Con t=4 qubits y φ=0.25, esperamos medir j=4 con probabilidad 1 (fase exactamente representable).

In [ ]:
from qiskit.primitives import StatevectorSampler

def run_qpe(phase: float, t: int, shots: int = 2048) -> dict:
    """Ejecuta QPE y retorna {fase_estimada: probabilidad}."""
    qc = qpe_circuit(phase, t)
    sampler = StatevectorSampler()
    job = sampler.run([qc], shots=shots)
    result = job.result()[0]
    counts = result.data.c.get_counts()

    fase_dist = {}
    for bitstr, cnt in counts.items():
        j = int(bitstr, 2)
        phi_est = j / (2 ** t)
        fase_dist[phi_est] = cnt / shots
    return fase_dist

# Fase exacta
dist = run_qpe(0.25, t=4)
print('φ = 0.25 (exacta):')
for phi, prob in sorted(dist.items(), key=lambda x: -x[1])[:5]:
    print(f'  φ̂ = {phi:.4f} con probabilidad {prob:.3f}')

# Fase inexacta (no representable exactamente con 4 bits)
dist2 = run_qpe(0.333, t=4)
print('\nφ = 0.333 (inexacta, distribución sobre vecinos):')
for phi, prob in sorted(dist2.items(), key=lambda x: -x[1])[:4]:
    print(f'  φ̂ = {phi:.4f} con probabilidad {prob:.3f}')

## 3. Precisión vs número de qubits de conteo

El error de estimación es ε ≤ 2π / 2^t para fase representable. Para fase arbitraria, la probabilidad de error ε < 1/2^{t-n} es al menos 1 - 1/2^{n-1}.

Incrementar t de 1 a 10 muestra cómo la distribución se concentra alrededor de la fase verdadera.

In [ ]:
import matplotlib.pyplot as plt

phase_true = 0.333
t_values = [3, 4, 5, 6, 7]
fig, axes = plt.subplots(1, len(t_values), figsize=(15, 4), sharey=False)

for ax, t in zip(axes, t_values):
    dist = run_qpe(phase_true, t=t, shots=4096)
    phis = sorted(dist.keys())
    probs = [dist[p] for p in phis]
    ax.bar(phis, probs, width=1/2**t * 0.8, color='steelblue', alpha=0.8)
    ax.axvline(phase_true, color='red', linestyle='--', label='φ real')
    ax.set_title(f't = {t} bits')
    ax.set_xlabel('φ estimada')
    ax.set_xlim(0.2, 0.5)

axes[0].set_ylabel('Probabilidad')
axes[0].legend()
plt.suptitle(f'QPE: precisión vs qubits de conteo (φ = {phase_true})', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Eigenvalores de una puerta T

La puerta T tiene matriz diag(1, e^{iπ/4}), así que:
- |0⟩ tiene fase φ=0
- |1⟩ tiene fase φ=1/8

QPE con t=3 bits puede representar 1/8 = 0.125 exactamente (j=1).

In [ ]:
def qpe_T_gate(eigenstate_label: str, t: int = 3) -> dict:
    """QPE para puerta T con eigenestado |0⟩ o |1⟩."""
    counting = QuantumRegister(t, 'count')
    eigenstate = QuantumRegister(1, 'eig')
    cr = ClassicalRegister(t, 'c')
    qc = QuantumCircuit(counting, eigenstate, cr)

    if eigenstate_label == '1':
        qc.x(eigenstate[0])

    qc.h(counting)

    for k in range(t):
        # T^{2^k} = Rz(π/4 * 2^k)
        for _ in range(2 ** k):
            qc.cp(np.pi / 4, counting[k], eigenstate[0])

    qft_inv = QFT(t, inverse=True, do_swaps=True)
    qc.append(qft_inv, counting)
    qc.measure(counting, cr)

    sampler = StatevectorSampler()
    job = sampler.run([qc], shots=1024)
    counts = job.result()[0].data.c.get_counts()
    return {int(k, 2) / 2**t: v / 1024 for k, v in counts.items()}

for label in ['0', '1']:
    dist = qpe_T_gate(label, t=3)
    top = max(dist, key=dist.get)
    phi_esperada = 0.0 if label == '0' else 1/8
    print(f'|{label}⟩: φ̂ = {top:.4f} (esperada={phi_esperada:.4f}) con prob={dist[top]:.3f}')

## 5. Resumen y complejidad

| Parámetro | Valor |
|-----------|-------|
| Qubits de conteo | t bits de precisión |
| Error máximo | ε ≤ 2π / 2^t |
| Puertas U controladas | O(2^t) en naive, O(t²) con QFT aproximada |
| Probabilidad de éxito | ≥ 1 − 1/(2^{n−1}) con t = n + extra bits |

**Aplicaciones directas**:
- **Algoritmo de Shor**: QPE sobre la puerta de multiplicación modular
- **HHL**: QPE para invertir autovalores de la matriz A
- **VQE**: QPE para medir la energía del Hamiltoniano

In [ ]:
# Resumen de precisión
print('Bits de conteo | Precisión máxima')
print('-' * 35)
for t in range(3, 11):
    precision = 1 / 2**t
    print(f'  t = {t:2d}         | Δφ ≤ {precision:.6f} ({100*precision:.3f}%)')